In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

## Data Selection

In [20]:
# Project path
DATA_PATH = Path("../data/raw/CarsData.csv")

cars = pd.read_csv(DATA_PATH)
print(f"The dataset has {cars.shape[0]} rows and {cars.shape[1]} columns.")


The dataset has 97712 rows and 10 columns.


In [21]:
cars.columns

Index(['model', 'year', 'price', 'transmission', 'mileage', 'fuelType', 'tax',
       'mpg', 'engineSize', 'Manufacturer'],
      dtype='str')

In [22]:
cars.info()

<class 'pandas.DataFrame'>
RangeIndex: 97712 entries, 0 to 97711
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   model         97712 non-null  str    
 1   year          97712 non-null  int64  
 2   price         97712 non-null  int64  
 3   transmission  97712 non-null  str    
 4   mileage       97712 non-null  int64  
 5   fuelType      97712 non-null  str    
 6   tax           97712 non-null  int64  
 7   mpg           97712 non-null  float64
 8   engineSize    97712 non-null  float64
 9   Manufacturer  97712 non-null  str    
dtypes: float64(2), int64(4), str(4)
memory usage: 7.5 MB


In [23]:
cars.head()

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Manufacturer
0,I10,2017,7495,Manual,11630,Petrol,145,60.10,1.00,hyundi
1,Polo,2017,10989,Manual,9200,Petrol,145,58.90,1.00,volkswagen
2,2 Series,2019,27990,Semi-Auto,1614,Diesel,145,49.60,2.00,BMW
3,Yeti Outdoor,2017,12495,Manual,30960,Diesel,150,62.80,2.00,skoda
4,Fiesta,2017,7999,Manual,19353,Petrol,125,54.30,1.20,ford


In [24]:
cars.describe()

,year,price,mileage,tax,mpg,engineSize
count,97712.00,97712.00,97712.00,97712.00,97712.00,97712.00
mean,2017.07,16773.49,23219.48,120.14,55.21,1.66
std,2.12,9868.55,21060.88,63.36,16.18,0.56
min,1970.00,450.00,1.00,0.00,0.30,0.00
25%,2016.00,9999.00,7673.00,125.00,47.10,1.20
50%,2017.00,14470.00,17682.50,145.00,54.30,1.60
75%,2019.00,20750.00,32500.00,145.00,62.80,2.00
max,2024.00,159999.00,323000.00,580.00,470.80,6.60


## Data Cleaning

In [25]:
# Check for missing values
missing_values = cars.isnull().sum()
missing_values

model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
Manufacturer    0
dtype: int64

In [35]:
# Check for duplicate rows
duplicate_count = cars.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


 No duplicates exist, so no need to remove 

In [27]:
# Standardise column names
cars = cars.rename(columns={
    "fuelType": "fuel_type",
    "engineSize": "engine_size",
    "Manufacturer": "manufacturer"
})

cars.columns

Index(['model', 'year', 'price', 'transmission', 'mileage', 'fuel_type', 'tax',
       'mpg', 'engine_size', 'manufacturer'],
      dtype='str')

In [28]:
for col in ["transmission", "fuel_type", "manufacturer"]:
    print(f"\n{col}:")
    print(cars[col].value_counts())


transmission:
transmission
Manual       55502
Semi-Auto    22296
Automatic    19905
Other            9
Name: count, dtype: int64

fuel_type:
fuel_type
Petrol      53982
Diesel      40419
Hybrid       3059
Other         246
Electric        6
Name: count, dtype: int64

manufacturer:
manufacturer
ford          17811
volkswagen    14893
vauxhall      13258
merc          12860
BMW           10664
Audi          10565
toyota         6699
skoda          6188
hyundi         4774
Name: count, dtype: int64


In [34]:
# Fix spelling 
cars["manufacturer"] = cars["manufacturer"].str.strip().str.title()

cars["manufacturer"] = cars["manufacturer"].replace({
    "Hyundi": "Hyundai",
    "Bmw": "BMW"
})

cars["manufacturer"].value_counts()

manufacturer
Ford          17811
Volkswagen    14893
Vauxhall      13258
Merc          12860
BMW           10664
Audi          10565
Toyota         6699
Skoda          6188
Hyundai        4774
Name: count, dtype: int64

In [37]:
CURRENT_YEAR = 2026

cars["car_age"] = CURRENT_YEAR - cars["year"]

cars[["year", "car_age"]].head()

,year,car_age
0,2017,9
1,2017,9
2,2019,7
3,2017,9
4,2017,9


In [39]:
# Check for missing columns
numeric_columns = ["year", "price", "mileage", "tax", "mpg", "engine_size", "car_age"]

cars[numeric_columns].describe()

,year,price,mileage,tax,mpg,engine_size,car_age
count,97712.00,97712.00,97712.00,97712.00,97712.00,97712.00,97712.00
mean,2017.07,16773.49,23219.48,120.14,55.21,1.66,8.93
std,2.12,9868.55,21060.88,63.36,16.18,0.56,2.12
min,1970.00,450.00,1.00,0.00,0.30,0.00,2.00
25%,2016.00,9999.00,7673.00,125.00,47.10,1.20,7.00
50%,2017.00,14470.00,17682.50,145.00,54.30,1.60,9.00
75%,2019.00,20750.00,32500.00,145.00,62.80,2.00,10.00
max,2024.00,159999.00,323000.00,580.00,470.80,6.60,56.00


In [40]:
print("Cars with price <= 0:", len(cars[cars["price"] <= 0]))
print("Cars with mileage < 0:", len(cars[cars["mileage"] < 0]))
print("Cars with mpg <= 0:", len(cars[cars["mpg"] <= 0]))
print("Cars with engine size <= 0:", len(cars[cars["engine_size"] <= 0]))
print("Cars with car age < 0:", len(cars[cars["car_age"] < 0]))

Cars with price <= 0: 0
Cars with mileage < 0: 0
Cars with mpg <= 0: 0
Cars with engine size <= 0: 268
Cars with car age < 0: 0


Cars with engine size 0 does not make sense, lets investigate

In [43]:
cars[cars["engine_size"] == 0]["fuel_type"].value_counts()

fuel_type
Petrol      158
Diesel       69
Hybrid       38
Electric      2
Other         1
Name: count, dtype: int64

Electric cars engine_size = 0 is ok beacuse they dont use the conventional engine, but in data, there are some eletric cars with engine_size.
<br>To avoid unfair comparisons, remove engine_size